#### **DPR**

In [1]:
# from src.retrievers.models.encoder import TransformerEncoder
from src.retrievers.dpr import DPR

from hydra import initialize, compose

with initialize(config_path="config", version_base=None):
    cfg = compose(config_name="config")

dpr = DPR(cfg)

/Users/rasmuslinnemann/Desktop/Courses/Semester7/02456 Deep Learning/Project/02456-retrieval-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1121 11:33:57.002000 2508 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4541.91it/s]
[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.s

In [2]:
print(dpr.query_encoder.model.config)               # BERT's full config

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.22.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}



In [56]:
input_query = dpr.query_encoder.tokenizer("dogs are cute", return_tensors="pt")

input_doc = dpr.doc_encoder.tokenizer("dogs are cute", return_tensors="pt")

In [57]:
output_query = dpr.query_encoder.model(**input_query)

output_doc = dpr.doc_encoder.model(**input_doc)

print(output_query.last_hidden_state.shape)
print(output_doc.last_hidden_state.shape)

torch.Size([1, 5, 768])
torch.Size([1, 5, 768])


In [58]:
output_query_pooled = output_query.pooler_output
output_doc_pooled = output_doc.pooler_output

print(output_query_pooled.shape)
print(output_doc_pooled.shape)

torch.Size([1, 768])
torch.Size([1, 768])


In [ ]:
output_query_pooled @ output_doc_pooled.T

tensor([[224.3565]], grad_fn=<MmBackward0>)

In [ ]:
print(dpr.query_encoder.model.config.hidden_size)   # hidden dim

768


In [ ]:
print(dpr.query_encoder.model)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

In [ ]:
# BertForPreTraining  =  body  +  MLM head  +  NSP head
# BertModel           =  body


In [ ]:
# query    → query BERT → pool → q  (768)
# document → doc BERT   → pool → d  (768)
# score = q · d          ← dot product; high means relevant


How does a Transformer Encoder work? What is BERT?

How to work with transformers in PyTorch


How to use contrastive learning for Dense Passage Retrieval.

What is self-attention and multi-head attention?


